# Assignment 5
In this assigment you will use Lambeq to extract the grammatical structure of sentences using a statistical parser and see the limits of such an approach. Then you will use these structured parses to solve a linguistic classification task.


## Introduction

Let us look at the below example sentence and see how Lambeq can parse this.
> "Colorless green ideas sleep furiously"

To assign the grammatical structures learnt in class to text, we will be using the Bobcat parser provided (formerly) by [Lambeq](https://docs.quantinuum.com/lambeq/intro.html).

In [ ]:
# !pip install numpy
from lambeq import BobcatParser
import os

In [ ]:
parser = BobcatParser(model_name_or_path=os.getcwd()+'/bobcat')

In Lambeq there are two ways to visualise a sentance. The first uses the parser's raw parse tree, coming from Combinatory Categorial Grammar (CCG), which for the purposes of this course can be understood as a kind of residuated monoid as seen in class (although **this is not strictly true**). Note that the convention for slashes in CCG is different from what is seen in class, so to avoid confusion we replace them with arrows. To see for yourself you can set 'use_slashes' to true.

In [ ]:
example_sentence = "Colorless green ideas sleep furiously"
tree = parser.sentence2tree(example_sentence)
print(tree.deriv(use_slashes=False))

As we now know, this can also be visualised as a diagram using pregroup grammar. You likely have a good intuition from the lectures and problem sheets regarding the similarities between the different formalisms, but the exact translation between them is more involved. For the curious, how this translation works and the details behind how CCG differs from what has been seen in class can be read about [here](https://aclanthology.org/2020.conll-1.24/) and [here](https://arxiv.org/abs/2105.07720).

In [ ]:
diagram = tree.to_diagram()
diagram.draw(figsize=(12,4))

## Section 1 [20 Marks]
---
### Problem 1 [6 Marks]

Parse the following sentences into residuated monoids and pregroups.

<ol>
    <li>Cute cats chasen ferocious mice.</li>
    <li>Cute cats chase mice in the garden.</li>
    <li>The cat chased and the mouse ate.</li>
    <li>The cat napped yesterday and today.</li>
    <li>The cat that chased the mouse.</li>
    <li>The cat that the mouse chased.</li>
</ol>

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

### Problem 2 [4 Marks]

Now try to parse the following sentences: 

<ol>
    <li>The old man the boat.</li>
    <li>As the woman read the magazine entertained the editors.</li>
</ol>

Verify if the provided parses are correct. If not, provide the correct parses yourself.

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->

### Problem 3 [4 Marks]

For the sentences from problem 2, force Lambeq to provide the correct parses by punctuating the sentences and provide them. State if you can't, and explain why this might be the case.

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->

### Problem 4 [6 Marks]

Help Lambeq in the same way you did in problem 3 to find the possible parses for the following ambiguous sentences: 

<ol>
    <li>Show me the meal on the flight from Ljubljana.</li>
    <li>Old men and women danced.</li>
</ol>

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->

# Section 2 [30 Marks]
---
## Problem 5 [14 Marks]

Now you will solve a basic classification task using Lambeq. We are working with the [AG News](https://huggingface.co/datasets/sh0416/ag_news) dataset, which contains titles and brief descriptions of news articles relating to four different topics: world, sports, business, and Sci/Tech. You will see that the original dataset along with a preprocessed smaller subset is provided. In the example code we work with the latter as this task can become intractable fast on a standard computer. In general, if you are not careful you might encounter issues with effiency, as hint for this section you might be interested in looking into diagram rewriting.

In [ ]:
import numpy as np
import pandas as pd
import torch

In [ ]:
df_train = pd.read_csv('data/train_data.csv')
df_test = pd.read_csv('data/test_data.csv')
print(df_train)

To accurately predict the class of a given sentence in this dataset, grammar won't be enough. We need to assign some sort of semantics to the words, this is your task in problem 5. In particular, you will learn tensors for the words using an [ansatz](https://docs.quantinuum.com/lambeq/tutorials/parameterise.html#classical-case), which takes a diagram and assigns a tensor space to it. You are free to use the original dataset to inflate the number of examples if you wish (within reason i.e. justifiable train-test split), although this is not necessary. However, it is certainly recommended for initial testing, at least, to use a fraction of the smaller supplied dataset.

In [ ]:
from lambeq import TensorAnsatz, AtomicType, PytorchModel, PytorchTrainer
from lambeq.backend.tensor import Dim
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

def accuracy(y_hat, y):
    return torch.sum(torch.eq(torch.round(torch.sigmoid(y_hat)), y))/len(y)/4

loss_fn = CrossEntropyLoss()

ansatz = TensorAnsatz({AtomicType.NOUN: Dim(4), AtomicType.SENTENCE: Dim(4)})

In [ ]:
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

train_data = [] 
val_data = [] 
test_data = [ansatz(diagram)] 

train_labels = []
val_labels = [] 
test_labels = torch.tensor([[1,0,0,0]])

model = PytorchModel.from_diagrams(train_data + val_data + test_data)

As far as learning goes you have three options: simply writing your own training loop as was seen in the week 7 guest lecture, defining your own PytorchModel subclass with a forward method as is done in PyTorch, or using the PytorchTrainer class defined in Lambeq.

In [ ]:
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

model.initialise_weights()

optimizer = Adam(model.parameters(), lr=1e-3)
for _ in range(5):
    optimizer.zero_grad()
    out = model.get_diagram_output([ansatz(diagram)])
    loss = loss_fn(out, torch.tensor([[1,0,0,0]], dtype=torch.float32))
    loss.backward()
    optimizer.step()
    print(loss)

print(f"Accuracy: {accuracy(model(test_data), test_labels)}")

Now visualise how well the model trained, and more specifically try visualising when the model performs poorly.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->

### Problem 6 [4 Marks]

Now try to improve the model, for example by playing with the different hyperparameters, or preferably otherwise. Keep in mind, the improvements can be in terms of efficiency or accuracy. Show the improvement and justify your choices.

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->

### Problem 7 [6 Marks]

Now that you have a reasonably well trained model, compare the tensorial representations of the following classes of words:

    a) Nouns   
    b) Adjectives   
    c) Verbs

Moreover, check the following:

    i) How similar are the nouns of the sentences with the same class? (For example, using cosine distance) Aggregate or visualise these in whichever way you see fit to demonstrate within class similarity.
    ii) Bonus: can you do this for adjectives and verbs aswell?

Note you will need to get the words along with the assigned parameters from the model.

In [ ]:
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''
print(model.symbols)
print(model.weights[3] @ model.weights[3])

<!-- Replace the contents of this cell with your written solution -->

## Problem 8 [6 Marks]

Repeat the classification task, but this time, use only the meanings of the nouns. In other words, try to predict the class of a sentence using only the learnt vectors of the nouns in the sentence. Once this is done, try adding the adjectives and verbs too. As a hint, this will require some creativity as a simple average is not possible here as the semantic representations of different words do not necessarily live in the same space.

In [ ]:
%%capture
''' 
    Replace the contents of this cell with your own code. Add more cells below if necessary.
'''

<!-- Replace the contents of this cell with your written solution -->